# 01 · Explore WESAD signals

Start with one participant file and inspect its contents before assuming anything about the local archive. WESAD uses Python pickle files; only load files from the official dataset archive because pickle loading can run code.

The chest and wrist devices record different channels at different rates. The notebook reads those names and shapes from the files. The labels are the study protocol conditions, not direct readings of how a participant felt.

## 1. Find the local dataset

Place extracted participant folders under `data/raw/WESAD/`, for example `data/raw/WESAD/S2/S2.pkl`. This cell discovers files. If none are present, it reports that exploration is pending and later data-dependent cells are skipped.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path.cwd() / "wearable-stress-ai", Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((candidate for candidate in candidates if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.wesad import find_subject_files, inspect_subject, LABEL_NAMES, SAMPLE_RATES_HZ

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "WESAD"
subject_files = find_subject_files(DATA_ROOT)
DATA_AVAILABLE = bool(subject_files)
print(f"Dataset root: {DATA_ROOT}")
if DATA_AVAILABLE:
    print(f"Found {len(subject_files)} participant files: {[p.parent.name for p in subject_files]}")
else:
    print("PENDING: no WESAD subject pickle files found. Obtain the dataset using data/README.md; no analysis or results are being claimed.")

## 2. Inspect participants, labels, channels and missing values

Each subject pickle contains synchronized labels and sensor data. The inspection table shows channel shapes, sample counts, documented rates where known, and non-finite values (NaN or infinity). For the published dataset, the label array shares the 700 Hz chest clock; wrist channels have their own slower rates. A channel length that differs from labels is expected for wrist data, not automatically a missing-data problem.

In [ ]:
if DATA_AVAILABLE:
    inventories = []
    subject_label_counts = []
    for path in subject_files:
        payload, rows = inspect_subject(path)
        inventories.extend(rows)
        labels = np.asarray(payload["label"]).reshape(-1)
        counts = pd.Series(labels).value_counts().sort_index()
        for label, count in counts.items():
            subject_label_counts.append({"subject": path.parent.name, "label": int(label),
                                         "label_name": LABEL_NAMES.get(int(label), f"other_{int(label)}"),
                                         "samples": int(count)})
    inventory = pd.DataFrame(inventories)
    display(inventory)
    display(pd.DataFrame(subject_label_counts).pivot_table(index="subject", columns="label_name", values="samples", fill_value=0))
    print("Unknown label codes:", sorted(set(np.concatenate([np.asarray(inspect_subject(p)[0]["label"]).reshape(-1) for p in subject_files]).astype(int)) - set(LABEL_NAMES)))

## 3. Plot short segments by labelled condition

WESAD is a collection of synchronized, segmented participant recordings—not one continuous recording. These plots select a short contiguous segment from one participant and one condition. The horizontal axis is seconds **within that participant recording**. We only plot chest ECG, EDA and temperature here because the published chest signals share a 700 Hz clock. ECG amplitude units are as stored by the sensor; consult the release documentation before assigning physical units. EDA is commonly reported in microsiemens and temperature in degrees Celsius, but confirm the exact scale in the supplied release notes.

In [ ]:
def find_key(mapping, desired):
    return next((k for k in mapping if str(k).casefold() == desired.casefold()), None)

def first_label_run(labels, wanted, min_samples):
    mask = np.asarray(labels).reshape(-1) == wanted
    starts = np.flatnonzero(mask & ~np.r_[False, mask[:-1]])
    for start in starts:
        end = start
        while end < len(mask) and mask[end]:
            end += 1
        if end - start >= min_samples:
            return start, end
    return None

if DATA_AVAILABLE:
    path = subject_files[0]
    payload, _ = inspect_subject(path)
    device = find_key(payload["signal"], "chest")
    if device is None:
        print("No chest key found; inspect the inventory and release notes before plotting.")
    else:
        chest = payload["signal"][device]
        labels = np.asarray(payload["label"]).reshape(-1)
        wanted_names = {1: "Baseline", 2: "Stress", 3: "Amusement"}
        fig, axes = plt.subplots(3, 1, figsize=(12, 8), constrained_layout=True)
        for ax, channel in zip(axes, ("ECG", "EDA", "TEMP")):
            key = find_key(chest, channel)
            if key is None:
                ax.set_visible(False)
                print(f"{channel} unavailable in {path.parent.name}; see inventory.")
                continue
            values = np.asarray(chest[key]).squeeze()
            if values.ndim != 1 or len(values) != len(labels):
                ax.set_visible(False)
                print(f"{channel} shape {np.asarray(chest[key]).shape} does not match label clock; not assuming alignment.")
                continue
            selected = None
            for label_id in wanted_names:
                selected = first_label_run(labels, label_id, 700 * 5)
                if selected is not None:
                    break
            if selected is None:
                ax.set_visible(False)
                print(f"No contiguous labelled segment available for {channel}.")
                continue
            start, end = selected
            stop = min(end, start + 700 * 10)
            t = np.arange(start, stop) / 700.0
            ax.plot(t, values[start:stop], linewidth=0.7)
            ax.set_title(f"{path.parent.name} chest {channel} · {wanted_names[int(labels[start])]} · 700 Hz")
            ax.set_xlabel("Seconds from recording start")
            ax.set_ylabel("Stored sensor value")
        plt.show()

## What to notice

- **Sampling frequency** is samples per second. At 700 Hz the chest recorder stores 700 points each second; wrist EDA and temperature are much slower at 4 Hz.
- **Amplitude** is the value on the vertical axis. Its physical unit depends on the channel and the sensor's documented scaling.
- **Axes** matter: wrist acceleration is three-axis, while several physiological channels are scalar. Do not treat different rates as if samples aligned one-to-one.
- Condition differences may reflect the protocol tasks, posture, movement, participants or sensors—not only stress.

### Glossary

- **Channel:** one measured signal, such as ECG or EDA.
- **Sampling rate:** number of recorded samples per second (Hz).
- **Label:** the protocol condition attached to samples.
- **Participant-independent:** validation where test participants do not appear in training.

### Exercises

1. Compare the number of samples per subject for chest and wrist channels. Why do they differ?
2. Plot another participant and compare the same condition. What varies between people?
3. Inspect the release's README for the sensor scaling and annotate a confirmed physical unit.
4. Check how many label codes appear. What does the release documentation say about codes outside 1–3?

**Status:** With no local files, this notebook can still teach the inspection workflow, but plots and measurements remain pending.